In [ ]:
##### Author Chamikara K


#####Chain is stitched together from left to right fashion, tied together via terminal H atoms forming linear chains
#####       H-Termer-nonmer-Initmer-H 
#####       <------ <------ <------


import pandas as pd
import os
import warnings

with warnings.catch_warnings():
    warnings.filterwarnings("ignore",category=DeprecationWarning)
    import pandas as pd
	
#INPUTS
repeats = input("Enter number of repeats: ")
grofile = input("Enter input gro file: ")
outfile = "nmer_chain.gro"
inputs = input("Enter X Y Z repeat distances in nanometers separated by a space: 0.0 0.0 1.53").split()
try:
    X, Y, Z = map(float, inputs) 
except ValueError:
    print("Please enter valid numbers.")
    exit()
	
import pandas as pd

def generate_polymer_gro(grofile, outfile, repeats, X, Y, Z):
    # Read the .gro file
    with open(grofile, "r") as gro:
        allines = gro.readlines()
    gro.close()

    # Parse the .gro file into a DataFrame
    grodirective = [line.split() for line in allines]
    grodirective_df = pd.DataFrame(grodirective[2:-1])  # Skip header lines and footer line

    # Initialize chain
    initmer = grodirective_df.iloc[:-1]
    nonmer = grodirective_df.iloc[1:-1]  # Split propagators

    newnonmer = nonmer.copy()
    polymer = initmer.copy()

    # Generate polymer chains based on repeats
    for i in range(1, repeats - 1):
        newnonmer.loc[:, 3] = nonmer.loc[:, 3].astype(float) + X * i
        newnonmer.loc[:, 4] = nonmer.loc[:, 4].astype(float) + Y * i
        newnonmer.loc[:, 5] = nonmer.loc[:, 5].astype(float) + Z * i
        polymer = pd.concat([polymer, newnonmer.astype(str)], ignore_index=False)

    polymer.iloc[:, 3:6] = polymer.iloc[:, 3:6].astype(float)

    # Process terminating fragment
    termer = grodirective_df.iloc[1:]  # Terminating fragment
    termer.iloc[:, 3:6] = termer.iloc[:, 3:6].astype(float)
    termernew = termer.copy()
    termernew.loc[:, 3] = termer.loc[:, 3] + X * (repeats - 1)
    termernew.loc[:, 4] = termer.loc[:, 4] + Y * (repeats - 1)
    termernew.loc[:, 5] = termer.loc[:, 5] + Z * (repeats - 1)

    polymer = pd.concat([polymer, termernew], ignore_index=True)
    polymer.iloc[:, 2] = pd.DataFrame([range(1, polymer.shape[0] + 1)]).T  # Fix atom order
    polymer.iloc[:, 3:6] = polymer.iloc[:, 3:6].astype(float)

    # Write output to *.gro file format
    with open(outfile, 'w+') as grofile:
        print('polymer gro file generated by polyGROgen | ChamikaraK', file=grofile)
        print(f'  {polymer.shape[0]}', file=grofile)
        for i in range(polymer.shape[0]):
            coord_string = "{}{}{}".format(format(polymer.iloc[i, 3], "8.3f"),
                                           format(polymer.iloc[i, 4], "8.3f"),
                                           format(polymer.iloc[i, 5], "8.3f"))
            site_string = "{0:>8s}{1:>7s}{2:>5d}{3}".format(polymer.iloc[i, 0],
                                                            polymer.iloc[i, 1],
                                                            polymer.iloc[i, 2],
                                                            coord_string)
            print(site_string, file=grofile)
        print('   10   10   10', file=grofile)

        

generate_polymer_gro(grofile, outfile, repeats, X, Y, Z)
